In [2]:
from google import genai
import os
from dotenv import load_dotenv, find_dotenv  # For loading .env file
from pydantic import BaseModel, Field
from typing import List, Tuple, Optional, Dict, Any, Literal, Annotated
from google.genai import types
import json
import random

from main.utils import *

2025-06-06 22:55:20.076 | INFO     | main.config:<module>:11 - PROJ_ROOT path is: D:\Git_repo\ViSoMMSD


In [2]:
key = os.getenv('API_KEY')
client = genai.Client(api_key= key)

In [3]:
for model in client.models.list():
    print(model.name)

models/embedding-gecko-001
models/gemini-1.0-pro-vision-latest
models/gemini-pro-vision
models/gemini-1.5-pro-latest
models/gemini-1.5-pro-001
models/gemini-1.5-pro-002
models/gemini-1.5-pro
models/gemini-1.5-flash-latest
models/gemini-1.5-flash-001
models/gemini-1.5-flash-001-tuning
models/gemini-1.5-flash
models/gemini-1.5-flash-002
models/gemini-1.5-flash-8b
models/gemini-1.5-flash-8b-001
models/gemini-1.5-flash-8b-latest
models/gemini-1.5-flash-8b-exp-0827
models/gemini-1.5-flash-8b-exp-0924
models/gemini-2.5-pro-exp-03-25
models/gemini-2.5-pro-preview-03-25
models/gemini-2.5-flash-preview-04-17
models/gemini-2.5-flash-preview-05-20
models/gemini-2.5-flash-preview-04-17-thinking
models/gemini-2.5-pro-preview-05-06
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-preview-image-generation
models/gemini-2.0-flash-lite-preview

In [6]:
import os
from google import genai
from google.genai import types
from yacs.config import CfgNode
import yaml
import re
from loguru import logger

def remove_emojis(text):
    # Unicode ranges cho emoji (cơ bản)
    emoji_pattern = re.compile(
        "[" 
        "\U0001F600-\U0001F64F"  # emoticons
        "\U0001F300-\U0001F5FF"  # symbols & pictographs
        "\U0001F680-\U0001F6FF"  # transport & map symbols
        "\U0001F1E0-\U0001F1FF"  # flags (iOS)
        "\U00002700-\U000027BF"
        "\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)


def remove_hashtags(text):
    return re.sub(r'#\w+', '', text)


def remove_links(text):
    # Loại bỏ http, https, www link
    return re.sub(r'http\S+|www\.\S+', '', text)


def preprocess_text(text):
    text = remove_emojis(text)
    text = remove_hashtags(text)
    text = remove_links(text)
    # Xóa khoảng trắng thừa
    text = ' '.join(text.split())
    return text


def get_config(yaml_file):
    return CfgNode(init_dict=yaml.load(open(yaml_file, "r"), Loader=yaml.FullLoader))


def label(gemini_api_key: str, model_name: str, input_data: dict, prompt: CfgNode):
    client = genai.Client(api_key=gemini_api_key,)
    model = model_name

    try:
        with open(input_data['image'], 'rb') as f:
            img_bytes = f.read()
    except Exception as e:
        logger.error(f"Error loading image {input_data.get('image')}: {e}")

    contents = [
        types.Content(
            role="user",
            parts=[
                types.Part.from_bytes(
                    mime_type="image/png",
                    data=img_bytes,
                ),
                types.Part.from_text(text=f"Caption: {preprocess_text(input_data['caption'])}. {prompt.user_prompt}"),
            ],
        ),
    ]

    generate_content_config = types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=genai.types.Schema(
            type = genai.types.Type.OBJECT,
            properties = {
                "label": genai.types.Schema(
                    type = genai.types.Type.STRING,
                ),
            },
        ),
        system_instruction=[
            types.Part.from_text(text=prompt.system_prompt),
        ],
    )

    result = client.models.generate_content(
        model=model,
        contents=contents,
        config=generate_content_config,
    )

    return result

In [11]:
import re
import json

def remove_json_markdown(response: str) -> dict:
    """
    Extracts JSON object from a markdown-style code block in LLM output.
    """
    match = re.search(r"```json\s*([\s\S]+?)\s*```", response)
    match2 = re.search(r"```\s*([\s\S]+?)\s*```", response)
    if match:
        json_str = match.group(1)
    elif match2:
        json_str = match2.group(1)
    else:
        json_str = response

    try:
        return json.loads(json_str)
    except Exception as e:
        print(f"Error parsing JSON: {e}")
        return None
    

def save_to_json(input_path, output_path, indent=4):
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(input_path, f, ensure_ascii=False, indent=indent)

In [17]:
gold =[]
pred =[]
for ith, item in enumerate(data):
    if "human_label" in item and "label" in item:
        gold.append(item["human_label"].strip().lower())
        pred.append(item["label"].strip().lower())
    else:
        item.update(remove_json_markdown(item['error']))



In [19]:
data[38]

{'caption': 'Ước gì có bồ như con cua khúm núm 😬 Ôm vợ chạy khắp thế gian.   Cụ thể là khi cảm thấy nguy hiểm, nó sẽ kẹp con cái dưới bụng sau đó chạy như bay, hoặc vùi con cái dưới cát để giấu =)))))  Điểm cộng: iu vợ, cõng vợ đi khắp nơi  Điểm trừ: thỉnh thoảng cõng nhầm vợ của thằng cua khác =)))))  #mcdymn',
 'image': 'D:/Git_repo/ViSoMMSD/data/all/merged\\fb_226.jpg',
 'human_label': 'Sarcasm',
 'error': '```json\n{"label": "sarcasm"}\n```',
 'label': 'sarcasm'}

In [20]:
save_to_json(data, 'gemma.json')

In [26]:
sarcasm_samples = [sample for sample in data if sample['label'] == 'Sarcasm']


KeyError: 'label'

In [25]:
pager = client.tunings.list(config={'page_size': 10})
print(pager.page_size)
print(pager[0])
pager.next_page()
print(pager[0])

10


IndexError: list index out of range

In [27]:
import json

with open(r'D:\Git_repo\ViSoMMSD\data\interim\full_data\multi.json', 'r', encoding='utf-8') as f:
    list_A = json.load(f)
with open(r'D:\Git_repo\ViSoMMSD\data\interim\full_data\llm_input.json', 'r', encoding='utf-8') as f:
    list_B = json.load(f)


In [29]:
list_B

[{'caption': 'Chán', 'image': 'fb_1004.jpg', 'human_label': 'Sarcasm'},
 {'caption': 'anh ăn lẩu xíu về liền',
  'image': 'fb_1013.jpg',
  'human_label': 'Sarcasm'},
 {'caption': 'Nhưng có gì cũng vẫn thích nói với mẹ, vòng lặp vô tận luôn mà =))',
  'image': 'fb_1028.jpg',
  'human_label': 'Sarcasm'},
 {'caption': 'Em sẽ nói ngắn gọn thôi, về cái chuyện mà anh rep tin nhắn chậm í, em cảm thấy:',
  'image': 'fb_1031.jpg',
  'human_label': 'Sarcasm'},
 {'caption': 'Cả lớp đều có giấy khen trừ m :((',
  'image': 'fb_1038.jpg',
  'human_label': 'Sarcasm'},
 {'caption': 'giới thiệu với m, đây là con của t.. giờ m nuôi nó đi',
  'image': 'fb_1041.jpg',
  'human_label': 'Sarcasm'},
 {'caption': 'Sự thât🙂', 'image': 'fb_1049.jpg', 'human_label': 'Sarcasm'},
 {'caption': 'nhưng mà họ không xem :((',
  'image': 'fb_1050.jpg',
  'human_label': 'Sarcasm'},
 {'caption': 'Ê nha :))', 'image': 'fb_106.jpg', 'human_label': 'Sarcasm'},
 {'caption': 'chưa đủ tró hả trời',
  'image': 'fb_1063.jpg',
  'h

In [17]:
import re

def preprocess_social_text(text):
    # Bỏ emoji
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
        "\U00002500-\U00002BEF"
        "\U00002700-\U000027BF"
        "\U000024C2-\U0001F251"
        "\U0001F900-\U0001F9FF"
        "\U0000200D"
        "\U00002300-\U000023FF"
        "\U0001FA70-\U0001FAFF"
        "\U0001F018-\U0001F270"
        "\U0001F650-\U0001F67F"
        "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r'', text)
    
    # Bỏ hashtag
    text = re.sub(r'#\w+', '', text)
    # Bỏ link
    text = re.sub(r'http\S+|www\.\S+', '', text)
    # Bỏ mention user
    text = re.sub(r'@\w+', '', text)
    # Chuẩn hóa khoảng trắng
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Ví dụ
text = "Xin chào các bạn 😄! Ghé thăm https://example.com #AI @user123"
print(preprocess_social_text(text))
# Output: Xin chào các bạn ! Ghé thăm


Xin chào các bạn ! Ghé thăm


In [54]:
import json

file1 = r'D:\Git_repo\ViSoMMSD\data\all\merged.json'        
file2 = r'D:\Git_repo\ViSoMMSD\data\interim\full_data\all.json'
output_file = "json_difference.json"

with open(file1, "r", encoding="utf-8") as f:
    data1 = json.load(f)
with open(file2, "r", encoding="utf-8") as f:
    data2 = json.load(f)

In [55]:
for each in data1:
    each['caption'] = preprocess_social_text(each['caption'])

for each in data2:
    each['caption'] = preprocess_social_text(each['caption'])

In [56]:
data1= [each for each in data1 if each['caption'].strip() != '']
data2= [each for each in data2 if each['caption'].strip() != '']

In [57]:
for each in data2:
    each['image'] = os.path.basename(each['image'])

In [33]:
image2_set = set(item['image'] for item in data2)

difference = [
    {"caption": item["caption"], "image": item["image"]}
    for item in data1
    if item['image'] not in image2_set
]

In [34]:
len(data1)

9544

In [35]:
len(data2)

948

In [105]:
save_to_json(data1, 'filter_raw.json')

In [36]:
len(difference)

8596

In [88]:
new = random.sample(difference, 1000-len(data2))


In [38]:
len(new)

52

In [90]:
for each in new:
    print(each['caption'])

Từ giờ mọi người có thể xếp hàng chào anh Thỏ bên dưới >:(
Aaaa
Khoa Pug ngồi cạnh 'diễn viên' ngôi sao nổi tiếng làng Hollywood Natalia Portman trong vlog mới đây, nhưng đặc biệt là anh Khoa chả biết cổ là ai
Hết thời
Ăn chay mùng 1 - Trần Bơm
Thẻn cồi - Trần Bơm
GẦN ĐÂY NHIỀU THUỐC GIẢ TRÀN LAN Càng lúc càng nhiều vụ hàng nhái – người dùng chỉ biết cầm thuốc về uống mà không hề biết xuất xứ nơi sản xuất sản phẩm. Long Châu là nhà thuốc đầu tiên chủ động thực hiện hàng loạt động tác minh bạch thông tin thuốc như in xuất xứ nơi sản xuất từng loại thuốc ngay trên hóa đơn. Một bước đơn giản nhưng cực kỳ cần thiết để minh bạch với người dân. Ai thường xuyên mua thuốc, nên giữ lại bill – kiểm tra lại thuốc mình đang dùng có rõ xuất xứ nơi sản xuất không nhé!
rớt tim luôn òi
=)))))))))))))))))))))))))))))) cre: observation's planet
Trời ơi dậy chưa đấy :)
Trung bình cmt tiktok tại các video diễu binh =)))))))))
‼ CHÁY RẤT TO TẠI AN KHÁNH, HOÀI ĐỨC Mong không có thiệt hại về người
Bà Nobi bắ

In [39]:
image_new = set(item['image'] for item in new)

In [40]:
len(image_new)

52

In [41]:
len(difference)

8596

In [ ]:


difference_new = [
    {"caption": item["caption"], "image": item["image"]}
    for item in difference
    if item['image'] not in image_new
]

In [43]:
len(difference_new)

8544

In [106]:
save_to_json(difference_new, 'remaining.json')

In [44]:
difference_new

[{'caption': 'Sau những ồn ào liên quan đến hành vi phạm pháp của bố vợ, Lee Seung Gi đã chính thức lên tiếng xin lỗi về những hệ lụy gây ra. Quyết định khó khăn nhất được anh đưa ra là đoạn tuyệt quan hệ với gia đình vợ sau nhiều đêm trăn trở. Ngày 29 vừa qua, Lee Seung Gi đã thông qua một tuyên bố bày tỏ sự hối lỗi sâu sắc liên quan đến Lee Hong Heon, bố dượng của bà xã Lee Da In. "Tôi vô cùng xin lỗi khi phải nói ra những lời này trong tâm trạng nặng trĩu," anh nghẹn ngào. Anh cho biết thêm, bố vợ anh đã từng bị kết án phạt tiền trong phiên phúc thẩm về các sai phạm trước đây, nhưng gần đây lại tiếp tục bị cơ quan điều tra khởi tố vì những hành vi tương tự. Sự việc này đồng nghĩa với việc ông Lee Hong Heon lại vướng vào một cáo buộc pháp lý mới, không liên quan đến vụ án trước. Lee Seung Gi chia sẻ: "Tôi đã đặt niềm tin vào mối quan hệ gia đình và chờ đợi kết quả, nhưng sự thật này khiến tôi vô cùng đau lòng." Về những phát ngôn có phần bênh vực bố vợ vào năm ngoái, Lee Seung Gi tự 

In [98]:
random_2k = random.sample(difference_new, 2000)

In [99]:
for each in random_2k:
    print(each['caption'])

PSG chính thức vô địch Ligue 1 sớm 6 vòng đấu
Ham hố lắm í =)))))
Sao biết vậy
Nhà ngta “bày” cỡ đó, nhà t “bay” cỡ này, coi ngang ngược hong
Tiên trách kỉ, hậu trách nhân
- kachiusa -
Cho ai chưa biết
Đấy cho nó đi ké sướng thế nó còn ngoạc mồm lên. Ăn được cái Tết hạnh phúc cũng có dễ đâu - Trần Bơm
SIUUUUUUUU 3-1 cho T1111111111111111111
đâu r tr
Tự nhiên lên 3tr followers rồi nên làm gì giờ mọi người
Nh mà nấu cái gì ra xá lợi cơ
NHÌN NGẮM HOÀ BÌNH CỦA BUỔI SÁNG HÔM NAY Ảnh: Jayni, Nguyễn Viết Thương
Giấc mơ nào cũng phải đến hồi kết
tóp tóp ít thôi
cảm xúc của tui sau khi đi xem diễu binh về
Xin chào mọi người, kachiusa đây, và chúc mừng sinh nhật lần thứ 7 của page! Đối với tôi thì con số 7 có ý nghĩa cực kì đặc biệt gắn liền với Meme tươi từ rừng Pác Bó, vì page đầu tiên được tạo ra ngày 17/7/2017, lúc tôi đang học lớp 7. Và ngày hôm nay, chúng ta cùng ăn mừng kỉ niệm 7 năm thành lập page. Chúng ta đã trải qua một hành trình dài, đã tạo ra hàng chục nghìn meme, và đã tạo dựng nê

In [100]:
random_2k

[{'caption': 'PSG chính thức vô địch Ligue 1 sớm 6 vòng đấu',
  'image': 'fb_355.jpg'},
 {'caption': 'Ham hố lắm í =)))))', 'image': 'fb_1722.jpg'},
 {'caption': 'Sao biết vậy', 'image': 'fb_2589.jpg'},
 {'caption': 'Nhà ngta “bày” cỡ đó, nhà t “bay” cỡ này, coi ngang ngược hong',
  'image': 'fb_6064.jpg'},
 {'caption': 'Tiên trách kỉ, hậu trách nhân', 'image': 'fb_2957.jpg'},
 {'caption': '- kachiusa -', 'image': 'fb_3717.jpg'},
 {'caption': 'Cho ai chưa biết', 'image': 'fb_6098.jpg'},
 {'caption': 'Đấy cho nó đi ké sướng thế nó còn ngoạc mồm lên. Ăn được cái Tết hạnh phúc cũng có dễ đâu - Trần Bơm',
  'image': 'fb_4096.jpg'},
 {'caption': 'SIUUUUUUUU 3-1 cho T1111111111111111111',
  'image': 'fb_7349.jpg'},
 {'caption': 'đâu r tr', 'image': 'fb_4560.jpg'},
 {'caption': 'Tự nhiên lên 3tr followers rồi nên làm gì giờ mọi người',
  'image': 'fb_8002.jpg'},
 {'caption': 'Nh mà nấu cái gì ra xá lợi cơ', 'image': 'fb_9453.jpg'},
 {'caption': 'NHÌN NGẮM HOÀ BÌNH CỦA BUỔI SÁNG HÔM NAY Ảnh: J

In [101]:
save_to_json(random_2k, '2k_sample.json')

In [2]:
gold = load_json(r'D:\Git_repo\ViSoMMSD\research\final\1k_sample_human_label.json')

In [10]:
text_label = [item['text_modality'].lower() for item in gold]
image_label = [item['image_modality'].lower() for item in gold]
multi_label = [item['multi_modality'].lower() for item in gold]

In [45]:
import pandas as pd
import matplotlib.pyplot as plt

# Đưa vào DataFrame cho tiện xử lý
df = pd.DataFrame({
    'text': text_label,
    'image': image_label,
    'multi': multi_label
})

# Đếm tần suất mỗi nhãn cho từng modality
counts = {
    'text': df['text'].value_counts(),
    'image': df['image'].value_counts(),
    'multi': df['multi'].value_counts()
}
counts

{'text': text
 non-sarcasm    949
 sarcasm         51
 Name: count, dtype: int64,
 'image': image
 non-sarcasm    746
 sarcasm        254
 Name: count, dtype: int64,
 'multi': multi
 non-sarcasm    673
 sarcasm        327
 Name: count, dtype: int64}

# Train Dev Test split

In [35]:
from sklearn.model_selection import train_test_split

# Đọc dữ liệu
gold = load_json(r'D:\Git_repo\ViSoMMSD\data\gold\1k_gold.json')

# Lấy nhãn
y = [each['text_modality'] for each in gold]

# Split lần 1: train 60%, remain 40%
train, remain, y_train, y_remain = train_test_split(
    gold, y, stratify=y, test_size=0.4, shuffle=True, random_state=42
)

# Split lần 2: dev 20%, test 20%
dev, test, y_dev, y_test = train_test_split(
    remain, y_remain, stratify=y_remain, test_size=0.5, shuffle=True, random_state=42
)

print(f"Train: {len(train)}, Dev: {len(dev)}, Test: {len(test)}")



Train: 600, Dev: 200, Test: 200


In [39]:
train =[
    {"caption": each['caption'],
     "label": each['text_modality']}
     for each in train
]

dev =[
    {"caption": each['caption'],
     "label": each['text_modality']}
     for each in dev
]

test =[
    {"caption": each['caption'],
     "label": each['text_modality']}
     for each in test
]

In [40]:
ith = 0
for each in train:
    if each['label'] == 'Non-sarcasm':
        ith += 1

print(ith)

569


In [41]:
ith = 0
for each in dev:
    if each['label'] == 'Non-sarcasm':
        ith += 1

print(ith)

190


In [42]:
ith = 0
for each in test:
    if each['label'] == 'Non-sarcasm':
        ith += 1

print(ith)

190


In [48]:
save_to_json(train, 'text_train.json')
save_to_json(dev, 'text_dev.json')
save_to_json(test, 'text_test.json')

In [77]:
gold = load_json(r'D:\Git_repo\ViSoMMSD\data\gold\1k_gold.json')

# LLM processing

In [81]:
check = load_json(r'D:\Git_repo\ViSoMMSD\research\final\2k_sample.json')


In [ ]:
text_1 = load_json(r'D:\Git_repo\ViSoMMSD\data\llm_label\1k_text_gemma-3-27b-it.json')
text_2 = load_json(r'D:\Git_repo\ViSoMMSD\data\llm_label\text_gemma-3-27b-it.json')

In [53]:
text_1_label = [each['text_llm_label'] for each in text_1] 

In [58]:
text_2_label = [each['text_llm_label'] for each in text_2] 

In [60]:
full_text = text_1 + text_2

In [63]:
full_text_label = [each['text_llm_label'] for each in full_text] 

In [66]:
save_to_json(full_text, 'text_llm_label.json')

In [65]:
len(full_text)

2000

In [82]:
check_text =[
    {"caption": each['caption'], "image": each['image']} for each in full_text
]

In [83]:
check == check_text

True

In [87]:
text_2[-1]

{'caption': '[Góc lịch sử] Ngày 16/8/1956, lần đầu tiên nhà máy Diêm Thống Nhất đón Bác về thăm. Bác đến thăm và dừng lại khá lâu ở các phân xưởng nan, dán, dầu thuốc và bao kiện. Đứng bên máy chặt que diêm tại phân xưởng nan, Bác chăm chú nhìn công nhân thao tác kĩ thuật và cầm trên tay mấy que rồi nói: - Các chú làm que diêm hơi to và dài (cỡ que diêm là 50mmx2mm). Đồng chí giám đốc đứng bên cạnh báo cáo: - Thưa Bác, nhân dân ta thường dùng diêm để hút thuốc lào nên phải làm to và dài ạ! Bác nói: - Nếu thế thì các chú nên làm hai loại. Bây giờ nhiều người dùng thuốc lá, làm que ngắn tiện bỏ túi mà lại tiết kiệm. Thực hiện lời dạy của Bác, sau đó ít lâu, que diêm được chỉnh lại ngắn và nhỏ hơn (40mmx1.6mm). ---------------------------- DIÊM THỐNG NHẤT – UY TÍN 67 NĂM VỀ DIÊM, BẬT LỬA & BAO BÌ CARTON Website: Đặt hàng: Danh sách cửa hàng:',
 'image': 'fb_6968.jpg',
 'text_llm_label': 'non-sarcasm'}

In [ ]:
image_1 = load_json(r'D:\Git_repo\ViSoMMSD\data\llm_label\1k_image_gemma-3-27b-it.json')
image_2 = load_json(r'D:\Git_repo\ViSoMMSD\data\llm_label\image_gemma-3-27b-it.json')

In [103]:
image = image_1 + image_2

In [ ]:
caption_set = set(item['caption'] for item in image)

diff_with_idx = [
    (i, item)
    for i, item in enumerate(check)
    if item['caption'] not in caption_set
]
diff_with_idx

[(1897,
  {'caption': 'Diana băng quần bảo vệ chị em trong giấc ngủ những ngày ấy và bảo vệ em khi dạo này chị ko còn lên cơn nữa.',
   'image': 'fb_7918.jpg'})]

In [118]:
diff_with_idx[0][1]['caption']

'Diana băng quần bảo vệ chị em trong giấc ngủ những ngày ấy và bảo vệ em khi dạo này chị ko còn lên cơn nữa.'

In [119]:
a = {'caption': 'Diana băng quần bảo vệ chị em trong giấc ngủ những ngày ấy và bảo vệ em khi dạo này chị ko còn lên cơn nữa.',
 'image': 'D:/Git_repo/ViSoMMSD/data/all/merged\\fb_7918.jpg',
 'image_llm_label': 'non-sarcasm'}

In [121]:
image.insert(1897, a)

In [129]:
failed = load_json(r'D:\Git_repo\ViSoMMSD\research\failed_image.json')

In [141]:
multi_1 = load_json(r'D:\Git_repo\ViSoMMSD\data\llm_label\1k_multi_gemma-3-27b-it.json')
multi_2 = load_json(r'D:\Git_repo\ViSoMMSD\data\llm_label\multi_gemma-3-27b-it.json')

In [142]:
multi = multi_1 + multi_2

In [135]:
len(multi)

1999

In [139]:
b=  {"caption": "Diana băng quần bảo vệ chị em trong giấc ngủ những ngày ấy và bảo vệ em khi dạo này chị ko còn lên cơn nữa.",
    "image": "fb_7918.jpg",
    "multi_llm_label": "non-sarcasm"}

In [144]:
multi.insert(1897, b)

In [187]:
caption_set = set(item['caption'] for item in multi)

diff_with_idx = [
    (i, item)
    for i, item in enumerate(check)
    if item['caption'] not in caption_set
]
diff_with_idx

[]

In [190]:
unlabel= [item 
         for item in multi
         if 'multi_llm_label' not in item
         ]

In [191]:
len(unlabel)

102

In [213]:
retry = load_json(r'D:\Git_repo\ViSoMMSD\data\external\llm_label\retry_multi_gemma-3-27b-it.json')

In [214]:
len(retry)

101

In [219]:
unlabel[34]

{'caption': 'Sauce: Huong Thu Chan (đăng trong Hang Pác Bó) - kachiusa -',
 'image': 'D:/Git_repo/ViSoMMSD/data/all/merged\\fb_3702.jpg'}

In [220]:
retry[34]

{'caption': 'Thời thế sao, kệ :v',
 'image': 'D:/Git_repo/ViSoMMSD/data/all/merged\\fb_792.jpg',
 'image_llm_label': 'sarcasm'}

In [221]:
c =   {
    "caption": "Sauce: Huong Thu Chan (đăng trong Hang Pác Bó) - kachiusa -",
    "image": "D:/Git_repo/ViSoMMSD/data/all/merged\\fb_3702.jpg",
    "image_llm_label": "sarcasm"
  }

In [222]:
retry.insert(34, c)

In [223]:
len(retry)

102

In [227]:
label= [item 
         for item in multi
         if 'multi_llm_label' in item
         ]

In [230]:
test = label + retry

In [231]:
len(test)

2000

In [233]:
save_to_json(test, 'multi_llm_label.json')

In [232]:
caption_set = set(item['caption'] for item in test)

diff_with_idx = [
    (i, item)
    for i, item in enumerate(check)
    if item['caption'] not in caption_set
]
diff_with_idx

[]

# EDA SILVER SET

In [71]:
text = load_json(r'D:\Git_repo\ViSoMMSD\data\silver\text_llm_label.json')
image = load_json(r'D:\Git_repo\ViSoMMSD\data\silver\image_llm_label.json')
multi = load_json(r'D:\Git_repo\ViSoMMSD\data\silver\multi_llm_label.json')

In [55]:
os.path.basename(multi[0]['image'])

'fb_355.jpg'

In [72]:
multi[0]

{'caption': 'PSG chính thức vô địch Ligue 1 sớm 6 vòng đấu',
 'image': 'D:/Git_repo/ViSoMMSD/data/all/merged\\fb_355.jpg',
 'multi_llm_label': 'non-sarcasm'}

In [73]:
multi = [
    {"caption": each['caption'],
     "image": os.path.basename(each['image']),
     "label": each['multi_llm_label']}
    for each in multi
]

In [75]:
len(multi)

2000

In [76]:
save_to_json(multi, 'silver_multi.json')

In [86]:
text_label = [item['text_llm_label'].lower() for item in text]
image_label = [item['image_llm_label'].lower() for item in image]
multi_label = [item['multi_llm_label'].lower() for item in multi]

In [87]:
import pandas as pd
import matplotlib.pyplot as plt

# Đưa vào DataFrame cho tiện xử lý
df = pd.DataFrame({
    'text': text_label,
    'image': image_label,
    'multi': multi_label
})

# Đếm tần suất mỗi nhãn cho từng modality
counts = {
    'text': df['text'].value_counts(),
    'image': df['image'].value_counts(),
    'multi': df['multi'].value_counts()
}
counts

{'text': text
 sarcasm        1051
 non-sarcasm     949
 Name: count, dtype: int64,
 'image': image
 sarcasm        1384
 non-sarcasm     616
 Name: count, dtype: int64,
 'multi': multi
 sarcasm        1019
 non-sarcasm     981
 Name: count, dtype: int64}